# safety_grid

> Phase 9 grids (attack, defence, monitoring, H3), the cell runner for `bench_grid.runGrid`, and the pre-registered H3 test (`PREREG_H3`).

Spec: `specs/multiagent/phase9/` (design-low-level section 5).

In [ ]:
#| default_exp safety_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib
import json

from explRL.safety_runs import SafetyConfig, runSafety

THETA = 2.5                     # strike threshold, calibrated in T7 (spec README)
PREREG_H3 = dict(version=1, study='h3', seeds=list(range(300, 330)), budget='compute', alpha=0.05, nflip=10000,
                 tests=dict(H3=dict(metric='loss', a='blackboard', b='messages', sizes=[8, 16, 32], sign=1)))
KINDS = ('liar', 'freerider', 'saboteur')
DEFENCES = (('trust', 0.0), ('reputation', 0.05), ('sanctions', 0.05))


def preregHashH3(prereg=PREREG_H3):
    return hashlib.sha256(json.dumps(prereg, sort_keys=True).encode()).hexdigest()[:12]


def counts(N):
    "Bad-agent counts: one, about 10% and about 25% of the team (at least one)."
    return (1, max(1, round(0.10 * N)), max(1, round(0.25 * N)))


def scell(grid, N, seed, B=15360.0, **kw):
    "A cell: `factor` = the grid, `level` = kind, number of bad agents, defence and q (so row keys stay distinct)."
    cfg = SafetyConfig(budget='compute', B=B, theta=THETA, **kw)
    return dict(study=grid, factor=grid, level=f'{cfg.kind}x{cfg.nBad}|{cfg.defence}|q={cfg.q}', N=N, seed=seed,
                runner='safety', cfg=cfg.asDict())


def _attacks(N, quick):
    return [('none', 0)] + [(k, n) for k in KINDS for n in ((1,) if quick else counts(N))]


def _setup(mode, sizes, seeds):
    return ((8,), (1000,), 1536.0) if mode == 'quick' else (sizes, seeds, 15360.0)


def attackGrid(mode='full'):
    "Every channel, every kind and count, no defence."
    sizes, seeds, B = _setup(mode, (16, 64), range(400, 410))
    chans = ('blackboard', 'messages') if mode == 'quick' else ('none', 'blackboard', 'messages', 'hierarchical')
    return [scell('attack', N, s, B, channel=ch, kind=k, nBad=n)
            for ch in chans for N in sizes for k, n in _attacks(N, mode == 'quick') for s in seeds]


def defenceGrid(mode='full'):
    "Blackboard and messages, each defence (trust q = 0; reputation and sanctions q = 0.05), every kind and count."
    sizes, seeds, B = _setup(mode, (16, 64), range(400, 410))
    if mode == 'quick':
        return [scell('defence', 8, 1000, B, channel='blackboard', kind=k, nBad=n, defence=d, q=0.05)
                for d, _ in DEFENCES for k, n in (('none', 0), ('liar', 1))]
    return [scell('defence', N, s, B, channel=ch, kind=k, nBad=n, defence=d, q=q)
            for ch in ('blackboard', 'messages') for d, q in DEFENCES for N in sizes for k, n in _attacks(N, False)
            for s in seeds]


def monitorGrid(mode='full'):
    "Blackboard, each defence at monitoring rates q = 0, 0.02, 0.05, 0.1, clean and with 25% liars."
    sizes, seeds, B = _setup(mode, (16, 64), range(400, 410))
    defs, qs = (('sanctions',), (0.05,)) if mode == 'quick' else (('trust', 'reputation', 'sanctions'), (0.0, 0.02, 0.05, 0.1))
    return [scell('monitor', N, s, B, channel='blackboard', kind=k, nBad=n, defence=d, q=q)
            for d in defs for q in qs for N in sizes for k, n in (('none', 0), ('liar', counts(N)[2])) for s in seeds]


def h3Grid(mode='full'):
    "H3: blackboard and messages, sizes 8, 16, 32, one liar vs clean, no defence (pilot seeds and budgets / 10 in quick)."
    sizes, seeds, B = ((8,), (1000, 1001), 1536.0) if mode == 'quick' else ((8, 16, 32), PREREG_H3['seeds'], 15360.0)
    return [scell('h3', N, s, B, channel=ch, kind=k, nBad=n)
            for ch in ('blackboard', 'messages') for N in sizes for k, n in (('none', 0), ('liar', 1)) for s in seeds]


def runSafetyCell(cell):
    "Run one Phase 9 cell (for `bench_grid.runGrid(cells, run=runSafetyCell)`)."
    row = runSafety(SafetyConfig.fromDict(cell['cfg']), cell['N'], cell['seed'])[3]
    row.update(study=cell['study'], factor=cell['factor'], level=cell['level'])
    return [row]

## Tests

In [ ]:
import time
import tempfile
from pathlib import Path
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.safety_grid import (PREREG_H3, preregHashH3, counts, attackGrid, defenceGrid, monitorGrid, h3Grid,
                                runSafetyCell)
from explRL.safety_runs import lossRows
from explRL.bench_grid import runGrid, writeCSV, readCSV, confirm, rowKey

test_eq((counts(16), counts(64), counts(8)), ((1, 2, 4), (1, 6, 16), (1, 1, 2)))
test_eq([len(g('full')) for g in (attackGrid, defenceGrid, monitorGrid, h3Grid)], [800, 1200, 480, 360])
assert all(300 <= c['seed'] <= 329 for c in h3Grid('full')) and all(400 <= c['seed'] <= 409 for c in attackGrid('full'))
allFull = attackGrid() + defenceGrid() + monitorGrid() + h3Grid()
keys = [rowKey(c | {'channel': c['cfg']['channel'], 'budget': c['cfg']['budget']}) for c in allFull]
test_eq(len(set(keys)), len(keys))                                                   # every cell has its own key

# the quick grids run in under 30 s and give identical CSVs for 1 and 4 workers
quick = attackGrid('quick') + defenceGrid('quick') + monitorGrid('quick') + h3Grid('quick')
t0 = time.perf_counter(); rows4 = runGrid(quick, 4, run=runSafetyCell); el = time.perf_counter() - t0
print(f'quick safety grids: {len(rows4)} rows in {el:.1f} s'); assert el < 30, el
rows1 = runGrid(quick, 1, run=runSafetyCell)
with tempfile.TemporaryDirectory() as td:
    a, b = writeCSV(rows4, Path(td) / 'a.csv'), writeCSV(rows1, Path(td) / 'b.csv')
    test_eq(a.read_bytes(), b.read_bytes())
    back = readCSV(a)
q = confirm(lossRows(back), PREREG_H3, seeds=(1000, 1001))
test_eq(q['H3']['n'], 2)

# confirm on synthetic loss rows with a known effect
rng = np.random.default_rng(0)
syn = [dict(study='h3', channel=ch, budget='compute', N=N, seed=s, loss=(0.06 if ch == 'blackboard' else 0.01) + rng.normal(0, 0.02))
       for s in PREREG_H3['seeds'] for N in (8, 16, 32) for ch in ('blackboard', 'messages')]
c = confirm(syn, PREREG_H3)['H3']
test_eq(c['verdict'], 'supported'); test_close(c['mean'], 0.05, eps=0.01); test_eq(len(preregHashH3()), 12)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()